# House Rent Prediction — Dự đoán giá thuê nhà bằng mô hình hồi quy

Notebook này xây dựng một pipeline Machine Learning hoàn chỉnh cho bài toán **dự đoán giá thuê nhà/căn hộ** từ dataset House Rent Prediction.

Bài toán thuộc nhóm **hồi quy có giám sát** vì biến mục tiêu `Rent` là giá trị số liên tục. Dataset có cả biến số như `BHK`, `Size`, `Bathroom` và biến phân loại như `City`, `Furnishing Status`, `Tenant Preferred`, `Point of Contact`, `Area Locality`.

Điểm quan trọng của notebook này là biến mục tiêu `Rent` có nhiều outlier và phân phối lệch phải. Vì vậy, thay vì huấn luyện trực tiếp trên `Rent`, notebook sử dụng `np.log1p(Rent)` để làm biến mục tiêu trong quá trình huấn luyện. Sau khi mô hình dự đoán xong, kết quả được chuyển ngược về giá thuê gốc bằng `np.expm1(...)` để đánh giá bằng các metric quen thuộc như MAE, RMSE và R².

Các mô hình được tối ưu bằng `GridSearchCV` ngay từ đầu gồm:

- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

Ngoài ra, cột `Area Locality` không bị bỏ hoàn toàn. Vì cột này có rất nhiều giá trị khác nhau, notebook sử dụng **Frequency Encoding** để giữ lại thông tin vị trí mà không tạo ra hàng nghìn cột như One-Hot Encoding.


## 1. Import thư viện

Cell này import toàn bộ thư viện cần dùng trong notebook.

Nhóm thư viện xử lý dữ liệu gồm `pandas` và `numpy`. `pandas` dùng để đọc file CSV, kiểm tra dữ liệu, xử lý cột và tạo bảng kết quả. `numpy` dùng cho các phép biến đổi số học, đặc biệt là `np.log1p()` và `np.expm1()` khi xử lý biến mục tiêu `Rent`.

Nhóm thư viện trực quan hóa gồm `matplotlib` và `seaborn`. Chúng được dùng để vẽ histogram, boxplot, scatter plot, barplot và heatmap correlation trong phần EDA.

Nhóm thư viện Machine Learning gồm `train_test_split`, `GridSearchCV`, `Pipeline`, `ColumnTransformer`, `OneHotEncoder`, `StandardScaler` và các mô hình hồi quy. Việc dùng `Pipeline` và `ColumnTransformer` giúp gom toàn bộ tiền xử lý và mô hình vào cùng một quy trình, hạn chế lỗi và tránh rò rỉ dữ liệu trong cross-validation.


In [ ]:
# Thư viện xử lý dữ liệu
import pandas as pd
import numpy as np

# Thư viện trực quan hóa dữ liệu
import matplotlib.pyplot as plt
import seaborn as sns

# Chia dữ liệu và GridSearchCV
from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline và tiền xử lý
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Các mô hình hồi quy
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Metric đánh giá hồi quy
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ẩn warning không cần thiết để notebook gọn hơn
import warnings
warnings.filterwarnings("ignore")


## 2. Đọc dữ liệu từ Google Drive

Cell này kết nối Google Colab với Google Drive và đọc file `House_Rent_Dataset.csv` từ thư mục Drive.

Đường dẫn đang dùng là:

```text
/content/drive/MyDrive/ML/BTL/Datasets/House_Rent_Dataset.csv
```

Sau khi đọc dữ liệu, notebook hiển thị 5 dòng đầu tiên bằng `data.head()` để kiểm tra dữ liệu đã được tải đúng hay chưa. Đây là bước đầu tiên cần làm trước mọi thao tác phân tích hoặc tiền xử lý.


In [ ]:
# Đọc dữ liệu từ file CSV
# Lưu ý: cần upload file House_Rent_Dataset.csv vào Google Colab trước khi chạy cell này
from google.colab import drive, files
drive.mount('/content/drive')
data = pd.read_csv("/content/drive/MyDrive/ML/BTL/Datasets/House_Rent_Dataset.csv")

# Hiển thị 5 dòng đầu tiên
data.head()


## 3. Kiểm tra nhanh cột `Point of Contact` bằng boxplot

Cột `Point of Contact` thể hiện người hoặc đơn vị cần liên hệ khi thuê nhà, ví dụ `Contact Owner`, `Contact Agent`, `Contact Builder`.

Về trực giác, cột này có vẻ không liên quan trực tiếp đến giá thuê. Tuy nhiên, trong dữ liệu thực tế, nó có thể phản ánh gián tiếp phân khúc nhà. Ví dụ, các căn hộ cao cấp có thể thường do môi giới đăng, trong khi nhà giá thấp hơn có thể do chủ nhà tự đăng.

Boxplot ở cell này giúp quan sát phân phối `Rent` theo từng nhóm `Point of Contact`. Nếu các nhóm có mức giá khác biệt rõ ràng, cột này có thể mang thông tin hữu ích cho mô hình và không nên bỏ ngay từ đầu.


In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=data,
    x="Point of Contact",
    y="Rent"
)

plt.show()


## 4. Giá thuê trung bình theo `Point of Contact`

Sau boxplot, cell này tính giá thuê trung bình của từng nhóm `Point of Contact` bằng `groupby()`.

Việc tính trung bình giúp định lượng rõ hơn sự khác biệt giữa các nhóm. Nếu `Contact Agent`, `Contact Owner` và `Contact Builder` có giá thuê trung bình rất khác nhau, điều đó cho thấy biến này có thể liên quan đến `Rent`.

Trong báo cáo, có thể giải thích rằng `Point of Contact` không phải nguyên nhân trực tiếp quyết định giá thuê, nhưng có thể là một biến đại diện cho phân khúc bất động sản hoặc cách đăng tin.


In [ ]:
data.groupby("Point of Contact")["Rent"].mean()


## 5. Kiểm tra tổng quan dataset

Trước khi tiền xử lý dữ liệu, cần nắm cấu trúc tổng quan của dataset.

Cell này thực hiện ba việc:

1. In ra kích thước dataset bằng `data.shape`, gồm số dòng và số cột.
2. Dùng `data.info()` để kiểm tra tên cột, kiểu dữ liệu và số lượng giá trị không null.
3. Dùng `data.describe()` để xem thống kê mô tả của các biến số như trung bình, độ lệch chuẩn, min, max và các phân vị.

Bước này giúp xác định dataset có những cột nào, kiểu dữ liệu ra sao, và biến mục tiêu `Rent` có phân phối giá trị rộng hay không.


In [ ]:
# Kiểm tra số dòng và số cột
print("Shape of dataset:", data.shape)

# Hiển thị thông tin cột, kiểu dữ liệu và số lượng non-null
print("Dataset information:")
data.info()

# Thống kê mô tả các biến số
print("Descriptive statistics:")
display(data.describe())


## 6. Kiểm tra missing values và duplicate values

Cell này kiểm tra hai vấn đề dữ liệu cơ bản:

- **Missing values**: giá trị bị thiếu ở từng cột.
- **Duplicate rows**: các dòng dữ liệu bị trùng lặp.

Dataset gốc được giả định là không có missing values, vì vậy notebook không dùng `SimpleImputer` trong pipeline. Tuy nhiên, vẫn cần kiểm tra bằng code để chứng minh điều này.

Nếu có dòng trùng lặp, notebook sẽ xóa bằng `drop_duplicates()`. Việc loại bỏ duplicate giúp tránh việc một số mẫu giống nhau bị học lặp lại quá nhiều, gây sai lệch kết quả đánh giá.


In [ ]:
# Kiểm tra số lượng giá trị null ở từng cột
missing_values = data.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Kiểm tra số dòng bị trùng lặp
duplicate_count = data.duplicated().sum()
print("Number of duplicated rows:", duplicate_count)

# Nếu có dòng trùng lặp thì xóa
if duplicate_count > 0:
    data = data.drop_duplicates()
    print("Duplicated rows removed.")

print("Shape after checking duplicates:", data.shape)


## 7. Kiểm tra các biến phân loại

Các cột dạng chữ không thể đưa trực tiếp vào mô hình hồi quy của scikit-learn. Do đó, cell này liệt kê các cột có kiểu dữ liệu `object` và kiểm tra số lượng giá trị khác nhau của từng cột.

Thông tin này rất quan trọng để chọn phương pháp mã hóa:

- Cột có ít giá trị khác nhau, ví dụ `City`, `Furnishing Status`, `Tenant Preferred`, có thể dùng One-Hot Encoding.
- Cột có quá nhiều giá trị khác nhau, ví dụ `Area Locality`, không nên One-Hot trực tiếp vì sẽ sinh ra quá nhiều cột và dễ overfit.

Kết quả từ cell này là cơ sở cho quyết định dùng Frequency Encoding với `Area Locality` ở các bước sau.


In [ ]:
# Lấy danh sách các cột phân loại dạng object
categorical_cols = data.select_dtypes(include="object").columns
print("Categorical columns:")
print(list(categorical_cols))

# Kiểm tra số lượng giá trị khác nhau trong từng cột phân loại
for col in categorical_cols:
    print("=" * 60)
    print(f"Column: {col}")
    print("Number of unique values:", data[col].nunique())
    print(data[col].value_counts().head(20))
    print()


## 8. Phân tích biến mục tiêu `Rent` và lý do dùng `log1p(Rent)`

`Rent` là biến mục tiêu cần dự đoán. Vì `Rent` là giá trị số liên tục nên đây là bài toán hồi quy.

Trong bài toán giá nhà hoặc giá thuê nhà, biến giá thường bị **lệch phải**: phần lớn mẫu có giá thuê thấp hoặc trung bình, nhưng có một số mẫu có giá thuê rất cao. Những giá trị rất cao này là outlier và có thể làm mô hình khó học hơn.

Cell này vẽ ba biểu đồ:

1. Histogram của `Rent` gốc để xem phân phối ban đầu.
2. Histogram của `log1p(Rent)` để xem phân phối sau khi log-transform.
3. Boxplot của `Rent` để kiểm tra outlier.

Việc dùng `np.log1p(Rent)` giúp giảm ảnh hưởng của các giá trị thuê cực lớn. Sau khi dự đoán, kết quả sẽ được chuyển ngược về thang giá thuê gốc bằng `np.expm1()` để đánh giá.


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data["Rent"], kde=True)
plt.title("Distribution of Rent")
plt.xlabel("Rent")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(np.log1p(data["Rent"]), kde=True, color="orange")
plt.title("Distribution of log1p(Rent)")
plt.xlabel("log1p(Rent)")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x=data["Rent"])
plt.title("Boxplot of Rent")
plt.xlabel("Rent")
plt.show()


## 9. Phân tích quan hệ giữa `Rent` và một số đặc trưng quan trọng

Cell này trực quan hóa mối quan hệ giữa `Rent` và các biến đầu vào quan trọng.

Các biểu đồ được sử dụng gồm:

- Scatter plot giữa `Size` và `Rent`: kiểm tra diện tích lớn hơn có thường đi kèm giá thuê cao hơn không.
- Boxplot giữa `BHK` và `Rent`: kiểm tra số phòng ảnh hưởng như thế nào đến giá thuê.
- Barplot giá thuê trung bình theo `City`: kiểm tra khác biệt giá thuê giữa các thành phố.
- Boxplot giữa `Furnishing Status` và `Rent`: kiểm tra nhà có nội thất đầy đủ có giá thuê cao hơn không.

Phần EDA này không chỉ giúp hiểu dữ liệu mà còn giúp giải thích vì sao các đặc trưng này được giữ lại trong mô hình.


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x="Size", y="Rent")
plt.title("Relationship between Size and Rent")
plt.xlabel("Size")
plt.ylabel("Rent")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=data, x="BHK", y="Rent")
plt.title("Rent by BHK")
plt.xlabel("BHK")
plt.ylabel("Rent")
plt.show()

plt.figure(figsize=(10, 5))
city_rent = data.groupby("City")["Rent"].mean().sort_values(ascending=False)
sns.barplot(x=city_rent.index, y=city_rent.values)
plt.title("Average Rent by City")
plt.xlabel("City")
plt.ylabel("Average Rent")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=data, x="Furnishing Status", y="Rent")
plt.title("Rent by Furnishing Status")
plt.xlabel("Furnishing Status")
plt.ylabel("Rent")
plt.show()


## 10. Ma trận tương quan giữa các biến số

Heatmap correlation giúp xem mức độ tương quan tuyến tính giữa các biến số trong dataset.

Một số biến như `BHK`, `Size`, `Bathroom` có thể có tương quan dương với `Rent`, nghĩa là khi số phòng, diện tích hoặc số phòng tắm tăng thì giá thuê cũng có xu hướng tăng.

Tuy nhiên, correlation chỉ phản ánh quan hệ tuyến tính. Giá thuê nhà thường chịu ảnh hưởng bởi các quan hệ phi tuyến và tương tác giữa nhiều biến, ví dụ `City` kết hợp với `Size` hoặc `Furnishing Status`. Vì vậy, heatmap chỉ là bước phân tích tham khảo, không đủ để kết luận mô hình nào sẽ tốt nhất.


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(data.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


## 11. Tiền xử lý cột `Posted On`

Cột `Posted On` là cột ngày đăng tin. Nếu giữ nguyên ở dạng chuỗi ngày tháng, mô hình sẽ không hiểu được ý nghĩa thời gian của nó.

Cell này chuyển `Posted On` sang kiểu `datetime`, sau đó tách thành ba đặc trưng mới:

- `Posted_Month`: tháng đăng tin.
- `Posted_Day`: ngày đăng tin.
- `Posted_DayOfWeek`: thứ trong tuần.

Sau khi tạo các đặc trưng mới, cột `Posted On` gốc được xóa. Cách xử lý này giúp mô hình sử dụng được thông tin thời gian dưới dạng số.


In [ ]:
# Chuyển cột Posted On sang dạng datetime
data["Posted On"] = pd.to_datetime(data["Posted On"])

# Tách ra các đặc trưng thời gian mới
data["Posted_Month"] = data["Posted On"].dt.month
data["Posted_Day"] = data["Posted On"].dt.day
data["Posted_DayOfWeek"] = data["Posted On"].dt.dayofweek

# Xóa cột Posted On gốc
data = data.drop("Posted On", axis=1)

# Kiểm tra kết quả
data.head()


## 12. Tiền xử lý cột `Floor`

Cột `Floor` chứa thông tin tầng hiện tại và tổng số tầng dưới dạng text, ví dụ:

```text
Ground out of 2
3 out of 5
10 out of 20
Upper Basement out of 4
```

Nếu đưa trực tiếp cột này vào One-Hot Encoding, mô hình sẽ mất ý nghĩa số học của tầng hiện tại và tổng số tầng. Vì vậy, notebook tách `Floor` thành hai cột số:

- `Floor_Level`: tầng hiện tại.
- `Total_Floors`: tổng số tầng của tòa nhà.

Quy ước mã hóa:

- `Ground` được chuyển thành `0`.
- `Upper Basement` được chuyển thành `-1`.
- `Lower Basement` được chuyển thành `-2`.

Sau khi tách xong, cột `Floor` gốc được xóa khỏi dataset.


In [ ]:
# Kiểm tra một số giá trị trong cột Floor trước khi xử lý
print(data["Floor"].unique()[:20])


def extract_floor_level(value):
    floor = str(value).split(" out of ")[0]

    if floor == "Ground":
        return 0
    elif floor == "Upper Basement":
        return -1
    elif floor == "Lower Basement":
        return -2
    else:
        try:
            return int(floor)
        except:
            return np.nan


def extract_total_floors(value):
    try:
        return int(str(value).split(" out of ")[1])
    except:
        return np.nan


# Tạo hai cột mới
data["Floor_Level"] = data["Floor"].apply(extract_floor_level)
data["Total_Floors"] = data["Floor"].apply(extract_total_floors)

# Xóa cột Floor ban đầu
data = data.drop("Floor", axis=1)

# Kiểm tra kết quả sau khi xử lý
data[["Floor_Level", "Total_Floors"]].head()


## 13. Kiểm tra lại missing values sau khi tạo đặc trưng mới

Mặc dù dataset gốc không có missing values, quá trình xử lý cột `Floor` có thể tạo ra `NaN` nếu gặp giá trị text bất thường không tách được.

Cell này kiểm tra lại số lượng missing values sau khi xử lý `Posted On` và `Floor`. Nếu có dòng phát sinh `NaN`, notebook xóa các dòng đó bằng `dropna()`.

Lý do không dùng `SimpleImputer` ở đây là vì các giá trị lỗi nếu có thường phát sinh từ quá trình chuyển đổi text và số lượng không nhiều. Việc xóa các dòng lỗi giúp giữ pipeline đơn giản và phù hợp với yêu cầu bài làm.


In [ ]:
# Kiểm tra null sau xử lý Posted On và Floor
print("Missing values after feature engineering:")
print(data.isnull().sum())

# Xóa dòng lỗi nếu có NaN phát sinh trong quá trình xử lý
before_dropna = data.shape[0]
data = data.dropna()
after_dropna = data.shape[0]

print("Rows before dropna:", before_dropna)
print("Rows after dropna:", after_dropna)
print("Rows removed:", before_dropna - after_dropna)


## 14. Kiểm tra cột `Area Locality`

`Area Locality` là biến vị trí chi tiết, ví dụ tên khu vực hoặc địa phương cụ thể. Đây là biến có thể rất quan trọng vì giá thuê nhà phụ thuộc mạnh vào vị trí.

Tuy nhiên, cột này có cardinality rất cao, tức là có rất nhiều giá trị khác nhau. Nếu dùng One-Hot Encoding trực tiếp, số lượng cột sau mã hóa sẽ tăng mạnh, làm mô hình phức tạp và dễ overfit.

Cell này chỉ kiểm tra số lượng giá trị khác nhau và hiển thị các khu vực xuất hiện nhiều nhất. Việc mã hóa Frequency Encoding sẽ được thực hiện sau khi chia train/test để tránh data leakage.


In [ ]:
# Kiểm tra số lượng giá trị khác nhau của Area Locality
print("Number of unique values in Area Locality:", data["Area Locality"].nunique())

# Xem những khu vực xuất hiện nhiều nhất
print("Top 10 most frequent Area Locality values:")
print(data["Area Locality"].value_counts().head(10))


## 15. Tách biến đầu vào `X` và biến mục tiêu `y`

Ở bước này, dataset được tách thành:

- `X`: toàn bộ đặc trưng đầu vào.
- `y`: biến mục tiêu cần dự đoán.

Thay vì dùng trực tiếp `Rent`, notebook dùng:

```python
y = np.log1p(data["Rent"])
```

Lý do là `Rent` có nhiều outlier và phân phối lệch phải. Log-transform giúp giảm ảnh hưởng của các giá trị cực lớn, làm quá trình học ổn định hơn.

Danh sách `numeric_features` bao gồm các biến số gốc và biến `Area_Locality_Freq`. Biến `Area_Locality_Freq` chưa được tạo ở cell này, nhưng sẽ được tạo ngay sau khi chia train/test. Việc đưa tên cột vào danh sách trước giúp pipeline được khai báo rõ ràng.


In [ ]:
# Biến mục tiêu
target = "Rent"

# X là các biến đầu vào, y là biến mục tiêu đã log-transform
X = data.drop(target, axis=1)
y = np.log1p(data[target])

numeric_features = [
    "BHK",
    "Size",
    "Bathroom",
    "Posted_Month",
    "Posted_Day",
    "Posted_DayOfWeek",
    "Floor_Level",
    "Total_Floors",
    "Area_Locality_Freq"
]

categorical_features = [
    "Area Type",
    "City",
    "Furnishing Status",
    "Tenant Preferred",
    "Point of Contact"
]

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Target transformation: y = log1p(Rent)")
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 16. Chia train/test và tạo Frequency Encoding cho `Area Locality`

Cell này thực hiện hai việc quan trọng.

Đầu tiên, dữ liệu được chia thành tập train và test theo tỉ lệ 80/20. Tập train dùng để huấn luyện và chạy GridSearchCV, còn tập test chỉ dùng để đánh giá cuối cùng.

Sau đó, `Area Locality` được mã hóa bằng Frequency Encoding. Điểm quan trọng là tần suất chỉ được tính trên `X_train`, không tính trên toàn bộ dataset. Đây là cách làm đúng để tránh **data leakage**.

Quy trình:

1. Tính tần suất xuất hiện của từng `Area Locality` trong tập train.
2. Map tần suất đó sang tập train và tập test.
3. Với locality trong test chưa từng xuất hiện trong train, gán giá trị `0`.
4. Xóa cột text `Area Locality` gốc sau khi đã tạo `Area_Locality_Freq`.

Cách này giữ lại một phần thông tin vị trí mà không làm tăng số chiều dữ liệu như One-Hot Encoding.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Frequency Encoding cho Area Locality chỉ dựa trên tập train để tránh leakage
area_locality_freq_map = X_train["Area Locality"].value_counts(normalize=True)

X_train = X_train.copy()
X_test = X_test.copy()

X_train["Area_Locality_Freq"] = X_train["Area Locality"].map(area_locality_freq_map)
X_test["Area_Locality_Freq"] = X_test["Area Locality"].map(area_locality_freq_map).fillna(0)

# Loại bỏ cột text gốc sau khi đã mã hóa tần suất
X_train = X_train.drop("Area Locality", axis=1)
X_test = X_test.drop("Area Locality", axis=1)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Area_Locality_Freq created from train-set frequencies.")
print(X_train[["Area_Locality_Freq"]].head())


## 17. Xây dựng pipeline tiền xử lý

Pipeline tiền xử lý gồm hai nhánh chính.

Nhánh biến số (`numeric_features`) dùng `StandardScaler` để chuẩn hóa dữ liệu. Việc chuẩn hóa giúp đưa các biến số về cùng thang đo, đặc biệt hữu ích với Linear Regression. Với các mô hình cây như Decision Tree, Random Forest và Gradient Boosting, scaling không bắt buộc, nhưng vẫn có thể giữ để pipeline thống nhất.

Nhánh biến phân loại (`categorical_features`) dùng `OneHotEncoder(handle_unknown="ignore")`. One-Hot Encoding chuyển các category thành dạng số để mô hình có thể học được. Tham số `handle_unknown="ignore"` giúp tránh lỗi nếu tập test có category chưa xuất hiện trong tập train.

Hai nhánh được kết hợp bằng `ColumnTransformer`, giúp xử lý đúng loại biến trong cùng một pipeline.


In [ ]:
# Pipeline cho biến số: chuẩn hóa các biến numeric, bao gồm cả Area_Locality_Freq
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# Pipeline cho biến phân loại: One-Hot Encoding cho các cột categorical còn lại
categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Kết hợp hai pipeline bằng ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


## 18. Hàm đánh giá mô hình

Hàm `evaluate_model()` nhận vào giá trị thật và giá trị dự đoán trên **thang giá thuê gốc**, sau đó tính bốn metric hồi quy:

- `MAE`: sai số tuyệt đối trung bình.
- `MSE`: sai số bình phương trung bình.
- `RMSE`: căn bậc hai của MSE, có cùng đơn vị với `Rent`.
- `R2`: mức độ mô hình giải thích được biến động của dữ liệu.

Trong notebook này, mô hình học trên `log1p(Rent)`, nhưng metric cuối cùng được tính sau khi chuyển ngược về `Rent` gốc. Điều này giúp kết quả dễ diễn giải hơn trong báo cáo.


In [ ]:
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2


## 19. Khai báo mô hình và không gian siêu tham số

Cell này khai báo bốn mô hình hồi quy và bộ siêu tham số cần thử cho từng mô hình.

`GridSearchCV` sẽ thử tất cả tổ hợp tham số trong mỗi `param_grid`, đánh giá bằng cross-validation và chọn bộ tham số có điểm tốt nhất.

Ý nghĩa một số tham số:

- `max_depth`: độ sâu tối đa của cây. Cây quá sâu dễ overfit, cây quá nông có thể underfit.
- `min_samples_split`: số mẫu tối thiểu để tách một node.
- `min_samples_leaf`: số mẫu tối thiểu ở một leaf node.
- `n_estimators`: số lượng cây trong Random Forest hoặc Gradient Boosting.
- `learning_rate`: tốc độ học trong Gradient Boosting.
- `subsample`: tỉ lệ mẫu dùng để huấn luyện từng bước boosting.
- `max_features`: số lượng đặc trưng được xem xét khi tách node trong Random Forest.

Linear Regression có ít siêu tham số nên chỉ thử `fit_intercept`.


In [ ]:
models_and_params = {
    "Linear Regression": {
        "model": LinearRegression(),
        "params": {
            "regressor__fit_intercept": [True, False]
        }
    },
    "Decision Tree Regressor": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            "regressor__max_depth": [None, 5, 10, 20],
            "regressor__min_samples_split": [2, 5, 10],
            "regressor__min_samples_leaf": [1, 2, 4]
        }
    },
    "Random Forest Regressor": {
    "model": RandomForestRegressor(random_state=42),
    "params": {
        "regressor__n_estimators": [100, 200, 300],
        "regressor__max_depth": [5, 10, 20, None],
        "regressor__min_samples_split": [2, 5, 10],
        "regressor__min_samples_leaf": [1, 2, 4],
        "regressor__max_features": ["sqrt", "log2", None]
    }
},
    "Gradient Boosting Regressor": {
    "model": GradientBoostingRegressor(random_state=42),
    "params": {
        "regressor__n_estimators": [100, 200, 300],
        "regressor__learning_rate": [0.01, 0.05, 0.1],
        "regressor__max_depth": [2, 3, 5],
        "regressor__subsample": [0.8, 1.0],
        "regressor__min_samples_leaf": [1, 2, 4]
    }
}
}


## 20. Huấn luyện mô hình bằng GridSearchCV

Mỗi mô hình được đặt trong một `Pipeline` gồm:

1. `preprocessor`: tiền xử lý biến số và biến phân loại.
2. `regressor`: mô hình hồi quy.

`GridSearchCV` được chạy với `cv=5`, nghĩa là tập train được chia thành 5 phần để đánh giá chéo. Điểm tối ưu là `R2` trên thang `log1p(Rent)`.

Sau khi chọn được mô hình tốt nhất trong từng nhóm, notebook dự đoán trên tập test. Vì mô hình dự đoán trên thang log, kết quả được chuyển ngược bằng `np.expm1()` trước khi tính MAE, MSE, RMSE và R² trên thang giá thuê gốc.

Việc đánh giá trên thang gốc giúp kết quả có ý nghĩa thực tế hơn, ví dụ MAE cho biết trung bình mô hình dự đoán sai bao nhiêu đơn vị tiền thuê.


In [ ]:
results = []
best_models = {}

for name, config in models_and_params.items():
    print("=" * 70)
    print(f"Training model on log1p(Rent): {name}")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", config["model"])
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=config["params"],
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_models[name] = best_model

    y_pred_log = best_model.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    y_test_original = np.expm1(y_test)

    mae, mse, rmse, r2 = evaluate_model(y_test_original, y_pred)

    results.append({
        "Model": name,
        "Best CV R2 (log Rent)": grid_search.best_score_,
        "Test MAE (Original Rent)": mae,
        "Test MSE (Original Rent)": mse,
        "Test RMSE (Original Rent)": rmse,
        "Test R2 (Original Rent)": r2,
        "Best Params": grid_search.best_params_
    })

    print("Best CV R2 (log Rent):", grid_search.best_score_)
    print("Best Params:", grid_search.best_params_)
    print("Test R2 (Original Rent):", r2)
    print("Test RMSE (Original Rent):", rmse)


## 21. So sánh kết quả các mô hình

Cell này tổng hợp kết quả của các mô hình vào một bảng và sắp xếp theo `Test R2 (Original Rent)` giảm dần.

Các cột quan trọng gồm:

- `Best CV R2 (log Rent)`: điểm R² tốt nhất trong cross-validation trên thang log.
- `Test MAE (Original Rent)`: sai số tuyệt đối trung bình trên giá thuê gốc.
- `Test MSE (Original Rent)`: sai số bình phương trung bình trên giá thuê gốc.
- `Test RMSE (Original Rent)`: căn bậc hai của MSE, dễ diễn giải vì cùng đơn vị với `Rent`.
- `Test R2 (Original Rent)`: khả năng giải thích biến động của giá thuê trên tập test.

Mô hình tốt thường có `Test R2` cao, đồng thời `Test MAE` và `Test RMSE` thấp.


In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Test R2 (Original Rent)", ascending=False)
results_df[[
    "Model",
    "Best CV R2 (log Rent)",
    "Test MAE (Original Rent)",
    "Test MSE (Original Rent)",
    "Test RMSE (Original Rent)",
    "Test R2 (Original Rent)"
]]


## 22. Xem bộ siêu tham số tốt nhất

Cell này hiển thị bộ tham số tốt nhất mà `GridSearchCV` tìm được cho từng mô hình.

Phần này nên được đưa vào báo cáo vì nó chứng minh rằng mô hình không được chọn thủ công, mà được tối ưu thông qua quá trình tìm kiếm siêu tham số có hệ thống.

Ví dụ, nếu Random Forest có `max_depth=10` và `min_samples_leaf=2`, có thể giải thích rằng cấu hình này giúp cân bằng giữa khả năng học dữ liệu và hạn chế overfitting.


In [ ]:
results_df[["Model", "Best Params"]]


## 23. Chọn mô hình tốt nhất và đánh giá cuối cùng

Sau khi so sánh kết quả, mô hình đứng đầu theo `Test R2 (Original Rent)` được chọn làm mô hình tốt nhất.

Cell này in ra:

- Tên mô hình tốt nhất.
- Bộ tham số tốt nhất.
- MAE, MSE, RMSE và R² cuối cùng trên tập test.

Đây là kết quả chính có thể dùng trong phần kết luận của báo cáo. Cần lưu ý rằng tập test chỉ được dùng ở bước đánh giá cuối cùng, không dùng để chọn tham số trong GridSearchCV.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = best_models[best_model_name]

print("Best Model:", best_model_name)
print("Best Parameters:")
print(results_df.iloc[0]["Best Params"])

y_pred_best_log = best_model.predict(X_test)
y_pred_best = np.expm1(y_pred_best_log)
y_test_original = np.expm1(y_test)

mae, mse, rmse, r2 = evaluate_model(y_test_original, y_pred_best)

print("Final Test MAE (Original Rent):", mae)
print("Final Test MSE (Original Rent):", mse)
print("Final Test RMSE (Original Rent):", rmse)
print("Final Test R2 (Original Rent):", r2)


## 24. Biểu đồ Actual vs Predicted

Biểu đồ này so sánh giá thuê thực tế và giá thuê mô hình dự đoán trên tập test.

Trục X là `Actual Rent`, tức giá thuê thật. Trục Y là `Predicted Rent`, tức giá thuê mô hình dự đoán.

Nếu mô hình dự đoán tốt, các điểm sẽ có xu hướng nằm gần đường chéo tưởng tượng từ góc dưới trái lên góc trên phải. Nếu nhiều điểm nằm rất xa, mô hình đang dự đoán sai đáng kể ở một số mẫu, thường là những căn hộ có giá thuê quá cao hoặc quá khác biệt so với phần lớn dữ liệu.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test_original, y_pred_best, alpha=0.6)
plt.xlabel("Actual Rent")
plt.ylabel("Predicted Rent")
plt.title(f"Actual vs Predicted Rent - {best_model_name}")
plt.show()


## 25. Residual Plot

Residual là sai số giữa giá trị thật và giá trị dự đoán:

```text
Residual = Actual Rent - Predicted Rent
```

Residual plot giúp kiểm tra xem sai số của mô hình có phân bố ngẫu nhiên hay có xu hướng rõ ràng.

Nếu residual phân bố tương đối ngẫu nhiên quanh đường 0, mô hình hoạt động ổn. Nếu residual tạo thành một hình dạng hoặc xu hướng rõ ràng, có thể mô hình chưa học hết quan hệ trong dữ liệu.

Trong bài toán giá thuê nhà, residual lớn thường xuất hiện ở các mẫu có giá thuê rất cao do outlier hoặc do dataset thiếu các đặc trưng quan trọng như vị trí chính xác, tiện ích, chất lượng nhà hoặc khoảng cách đến trung tâm.


In [ ]:
residuals = y_test_original - y_pred_best

plt.figure(figsize=(8, 6))
plt.scatter(y_pred_best, residuals, alpha=0.6)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Rent")
plt.ylabel("Residuals")
plt.title(f"Residual Plot - {best_model_name}")
plt.show()


## 26. Kết luận chung

Notebook đã xây dựng một pipeline hoàn chỉnh cho bài toán dự đoán giá thuê nhà.

Các điểm chính của bài làm:

1. `Rent` là biến mục tiêu liên tục nên bài toán thuộc nhóm hồi quy.
2. `Rent` có nhiều outlier và phân phối lệch phải, vì vậy notebook dùng `log1p(Rent)` để huấn luyện mô hình ổn định hơn.
3. Cột `Posted On` được tách thành các đặc trưng thời gian.
4. Cột `Floor` được tách thành `Floor_Level` và `Total_Floors`.
5. Cột `Area Locality` có quá nhiều giá trị khác nhau nên được xử lý bằng Frequency Encoding thay vì One-Hot Encoding trực tiếp.
6. Các biến phân loại còn lại được mã hóa bằng One-Hot Encoding.
7. Các biến số được chuẩn hóa bằng StandardScaler.
8. Bốn mô hình hồi quy được tối ưu bằng GridSearchCV ngay từ đầu.
9. Kết quả cuối cùng được đánh giá trên thang giá thuê gốc bằng MAE, MSE, RMSE và R².

Hướng cải thiện trong tương lai:

- Thử thêm ExtraTreesRegressor, HistGradientBoostingRegressor, XGBoost, LightGBM hoặc CatBoost.
- Xử lý outlier cực đoan bằng percentile hoặc IQR.
- Thử Target Encoding cho `Area Locality`.
- Thu thập thêm đặc trưng về vị trí, tiện ích, khoảng cách đến trung tâm, giao thông và chất lượng căn hộ.
